# ET false-positive failure cases (Swish under BN / GN)

CPU runtime is enough. Reads the per-case Synapse CSVs (`workspace/Synapse (Results)/`) to pick cases, the submission zips (`workspace/BraTS2023_Submission_500ep_*.zip`) for the predictions, and the raw validation T1ce for the background. Writes `workspace/Figures/failure_cases_ET_false_positives.png` and prints a caption with the Synapse FP counts.

Label scheme: 1 = NCR (red), 2 = ED (green), 3 = ET (yellow). No ground truth is shown — the validation labels are sequestered — so the panel shows *predicted* ET lesions absent from the baseline prediction, and the caption quotes the number of ET lesions Synapse scored as false positives for each model on that case.

In [ ]:
from google.colab import drive
import os, io, zipfile, glob
drive.mount('/content/drive', force_remount=False)
WORKSPACE = "/content/drive/MyDrive/nnU-Net Project/workspace"
CSV_DIR   = os.path.join(WORKSPACE, "Synapse (Results)")
RAW_VAL   = os.path.join(WORKSPACE, "ASNR-MICCAI-BraTS2023-GLI-Challenge-ValidationData")
FIG_DIR   = os.path.join(WORKSPACE, "Figures"); os.makedirs(FIG_DIR, exist_ok=True)

MODELS = {
    "IN + LeakyReLU": ("basic_LeakyReLU.csv", "BraTS2023_Submission_500ep_LeakyReLU.zip"),
    "GN + LeakyReLU": ("GN_LeakyReLU.csv",    "BraTS2023_Submission_500ep_GN_LeakyReLU.zip"),
    "BN + Swish":     ("BN_Swish.csv",        "BraTS2023_Submission_500ep_BN_Swish.zip"),
    "GN + Swish":     ("GN_Swish.csv",        "BraTS2023_Submission_500ep_GN_Swish.zip"),
}
for k,(c,z) in MODELS.items():
    for p in (os.path.join(CSV_DIR,c), os.path.join(WORKSPACE,z)):
        print(("[ok]   " if os.path.exists(p) else "[MISSING] ") + p)
assert os.path.isdir(RAW_VAL), RAW_VAL


In [ ]:
import pandas as pd, numpy as np
def load(csv):
    d = pd.read_csv(os.path.join(CSV_DIR, csv), index_col=0); return d[d.index.str.startswith("BraTS")]
D = {k: load(c) for k,(c,_) in MODELS.items()}
idx = D["IN + LeakyReLU"].index
for k,v in D.items(): assert len(v)==219 and (v.index==idx).all(), k

fp = pd.DataFrame({k: v["Num_FP_ET"] for k,v in D.items()})
et = pd.DataFrame({k: v["LesionWise_Dice_ET"] for k,v in D.items()})
cand = fp[(fp["IN + LeakyReLU"]==0) & (fp["BN + Swish"]>=1) & (fp["GN + Swish"]>=1)].copy()
cand["extra"] = cand["BN + Swish"] + cand["GN + Swish"]
cand["ET_dice_IN"] = et.loc[cand.index, "IN + LeakyReLU"]
cand = cand[cand.ET_dice_IN >= 0.7].sort_values(["extra","ET_dice_IN"], ascending=[False, False])
print(f"{len(cand)} candidate cases (baseline FP_ET = 0, both Swish cells >= 1, baseline ET Dice >= 0.7)")
print(cand.head(10))
CASES = list(cand.index[:4])
print("\nSelected:", CASES)


In [ ]:
import nibabel as nib, gzip
from scipy import ndimage
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Ellipse

_zips = {k: zipfile.ZipFile(os.path.join(WORKSPACE, z)) for k,(_,z) in MODELS.items()}
def pred(model, case):
    zf = _zips[model]; name = next(n for n in zf.namelist() if n.endswith(f"{case}.nii.gz"))
    with zf.open(name) as f: data = f.read()
    if name.endswith(".gz"): data = gzip.decompress(data)
    return np.asanyarray(nib.Nifti1Image.from_bytes(data).dataobj).astype(np.uint8)
def t1c(case):
    p = glob.glob(os.path.join(RAW_VAL, case, f"{case}-t1c.nii.gz")) or glob.glob(os.path.join(RAW_VAL, case, "*t1c*.nii.gz")) or glob.glob(os.path.join(RAW_VAL, case, "*t1ce*.nii.gz"))
    return np.asanyarray(nib.load(p[0]).dataobj).astype(np.float32)

def extra_et_components(p_model, p_base):
    lab, n = ndimage.label(p_model == 3); base_et = p_base == 3; comps = []
    for i in range(1, n+1):
        m = lab == i
        if not (m & base_et).any(): comps.append((m.sum(), m))
    return sorted(comps, key=lambda t: -t[0])

CMAP = ListedColormap([(0,0,0,0), (0.85,0.1,0.1,0.85), (0.1,0.7,0.1,0.75), (1.0,0.85,0.0,0.95)])
cols = list(MODELS)
fig, axes = plt.subplots(len(CASES), len(cols)+1, figsize=(2.6*(len(cols)+1), 2.7*len(CASES)))
caption_rows = []
for r, case in enumerate(CASES):
    P = {k: pred(k, case) for k in cols}; img = t1c(case)
    comps = extra_et_components(P["GN + Swish"], P["IN + LeakyReLU"]) or extra_et_components(P["BN + Swish"], P["IN + LeakyReLU"])
    if comps:
        m = comps[0][1]; z = int(np.bincount(np.where(m)[2]).argmax()); cx, cy, cz = [float(v) for v in ndimage.center_of_mass(m)]
    else:
        z = int(np.argmax((P["GN + Swish"]==3).sum(axis=(0,1)))); cx = cy = None
    lo, hi = np.percentile(img[img>0], [1, 99.5]); sl = np.clip((img[:,:,z]-lo)/(hi-lo+1e-6), 0, 1)
    def show(ax, overlay=None, title=""):
        ax.imshow(np.rot90(sl), cmap="gray", vmin=0, vmax=1)
        if overlay is not None: ax.imshow(np.rot90(overlay[:,:,z]), cmap=CMAP, vmin=0, vmax=3, interpolation="nearest")
        ax.set_title(title, fontsize=8); ax.axis("off")
    show(axes[r,0], None, f"{case}\nT1ce, slice {z}")
    for c, k in enumerate(cols, start=1):
        show(axes[r,c], P[k], f"{k}\nFP ET (Synapse) = {int(fp.loc[case,k])}")
        if cx is not None and k in ("BN + Swish", "GN + Swish"):
            H, W = sl.shape  # rot90 maps array (x,y) -> display col=y, row=W-1-x
            axes[r,c].add_patch(Ellipse((cx, W-1-cy), 30, 30, fill=False, ec="cyan", lw=1.6))  # array (x,y) -> rot90 display (col=x, row=Y-1-y)
    # crop every panel of this row to the tumour (union of all predictions on this slice), padded
    U = np.zeros_like(sl, dtype=bool)
    for k in cols: U |= P[k][:,:,z] > 0
    xs, ys = np.where(U); Y = sl.shape[1]; pad = 28
    if len(xs):
        x0, x1 = max(xs.min()-pad, 0), min(xs.max()+pad, sl.shape[0]-1); y0, y1 = max(ys.min()-pad, 0), min(ys.max()+pad, Y-1)
        side = max(x1-x0, y1-y0); cxm, cym = (x0+x1)/2, (y0+y1)/2
        for ax in axes[r]:
            ax.set_xlim(cxm-side/2, cxm+side/2); ax.set_ylim(Y-1-cym+side/2, Y-1-cym-side/2)
    caption_rows.append(f"{case}: ET FP lesions scored by Synapse — " + ", ".join(f"{k} {int(fp.loc[case,k])}" for k in cols) + f"; baseline ET Dice {et.loc[case,'IN + LeakyReLU']:.3f}")
plt.tight_layout(h_pad=0.6, w_pad=0.2)
out = os.path.join(FIG_DIR, "failure_cases_ET_false_positives.png"); plt.savefig(out, dpi=300, bbox_inches="tight"); print("saved", out)
from IPython.display import Image, display; display(Image(out, width=1100))
print("\nCAPTION DATA\n" + "\n".join(caption_rows))


## Caption (draft)

FIGURE X. Enhancing Tumour false-positive failure cases. Each row is one BraTS 2023 validation case (T1ce, axial slice through the lesion); columns show the predicted segmentation of IN + LeakyReLU (baseline), GN + LeakyReLU, BN + Swish and GN + Swish (red = NCR, green = ED, yellow = ET). Cyan circles mark ET lesions predicted by the Swish models that are absent from the baseline prediction. Reference segmentations are sequestered by the challenge; the number of ET lesions scored as false positives by the Synapse lesion-wise pipeline is given above each panel. Cases were selected as those with no baseline ET false positive, at least one under both BN + Swish and GN + Swish, and baseline ET Dice ≥ 0.7, ranked by the number of extra lesions.